# Nextbike Brno Availability Baseline

Generated at `2026-05-25T09:43:47+02:00`.

This notebook reproduces the first 30-minute station emptiness baseline/model analysis.


## Current Summary

| Metric | Value |
|---|---|
| `collection_runs` | `793` |
| `first_collected_at` | `2026-05-24 18:32:12+02:00` |
| `latest_collected_at` | `2026-05-25 09:43:30+02:00` |
| `station_rows` | `235521` |
| `free_bike_rows` | `434129` |
| `distinct_stations` | `297` |
| `bikes_available_min_avg_max` | `542/564.49/573` |
| `db_size_mb` | `24.76` |

## Dataset Summary

| Metric | Value |
|---|---|
| `rows` | `224829` |
| `stations` | `297` |
| `first_collected_at` | `2026-05-24 20:17:28+02:00` |
| `latest_collected_at` | `2026-05-25 09:12:36+02:00` |
| `empty_future_rate` | `0.3974` |
| `avg_abs_change` | `0.088` |
| `changed_rate` | `0.0699` |
| `null_lag_5m` | `2970` |
| `null_lag_15m` | `6831` |

Build settings:

- horizon minutes: `30`
- max target delay minutes: `40`
- lag tolerance minutes: `10`


In [ ]:
from pathlib import Path
import duckdb
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'data' / 'nextbike.duckdb').exists():
    ROOT = ROOT.parent
DB_PATH = ROOT / 'data' / 'nextbike.duckdb'
TABLE_NAME = 'training_station_availability'

con = duckdb.connect(str(DB_PATH), read_only=True)
df = con.sql(f"select * from {TABLE_NAME} order by collected_at, station_id").df()
df.head()

In [ ]:
df[['bikes_now', 'bikes_future', 'empty_now', 'empty_future']].describe(include='all')

## Baseline Metrics

| Model | Accuracy | Precision | Recall | F1 | ROC AUC | Avg precision | Pred empty rate |
|---|---:|---:|---:|---:|---:|---:|---:|
| `persistence_empty_now` | 0.9584 | 0.9603 | 0.9394 | 0.9497 | 0.9558 | 0.9274 | 0.4086 |
| `station_prior_empty_rate` | 0.8773 | 0.8843 | 0.8125 | 0.8469 | 0.9194 | 0.8906 | 0.3838 |
| `low_inventory_le_1` | 0.8110 | 0.6912 | 0.9898 | 0.8140 | 0.9715 | 0.9412 | 0.5982 |
| `majority_train_class` | 0.5823 | 0.0000 | 0.0000 | 0.0000 | 0.5000 | 0.4177 | 0.0000 |

In [ ]:
from nextbike_analysis.modeling import evaluate_baselines

baseline = evaluate_baselines(
    db_path=DB_PATH,
    table_name=TABLE_NAME,
    test_fraction=0.25,
    low_bike_threshold=1,
)
[(row.name, row.f1, row.roc_auc) for row in baseline.metrics]

## Model Metrics

| Model | Accuracy | Precision | Recall | F1 | ROC AUC | Avg precision | Pred empty rate |
|---|---:|---:|---:|---:|---:|---:|---:|
| `hist_gradient_boosting` | 0.9566 | 0.9557 | 0.9398 | 0.9477 | 0.9804 | 0.9681 | 0.4108 |
| `logistic_regression` | 0.9406 | 0.9273 | 0.9308 | 0.9291 | 0.9782 | 0.9644 | 0.4193 |

In [ ]:
from nextbike_analysis.modeling import train_models

model_result, _ = train_models(
    db_path=DB_PATH,
    table_name=TABLE_NAME,
    test_fraction=0.25,
    model_dir=None,
)
[(row.name, row.f1, row.roc_auc) for row in model_result.metrics]

## Notes

The current sample is mostly overnight, so persistence is a very strong baseline.
Do not judge advanced model classes until we have morning/daytime demand in the dataset.
